# Sionna RT → OCUDU RAN

A ray-traced Munich channel drives a live 5G stack. This notebook shows the scene, the
route used to move the receiver, and the resulting propagation.

Run the cells in order.

In [ ]:
import json
import os

import numpy as np
import matplotlib.pyplot as plt

import sionna.rt as rt
from sionna.rt import (Camera, PathSolver, PlanarArray, Receiver, Transmitter,
                       load_scene)

REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
TX_POS = [8.5, 21.0, 27.0]      # gNB on a rooftop
RX_START = [20.0, 40.0, 1.5]    # UE at street level, near the gNB
RX_END = [20.0, 300.0, 1.5]     # UE far down the street
SRATE = 61.44e6
NUM_TAPS = 16
print("sionna.rt", rt.__version__)

## 1. The scene

Munich, with the gNB (blue) and the UE (green).

In [ ]:
scene = load_scene(rt.scene.munich)
scene.tx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.rx_array = PlanarArray(num_rows=1, num_cols=1, pattern="iso", polarization="V")
scene.add(Transmitter(name="tx", position=TX_POS))
rx = Receiver(name="rx", position=RX_START)
scene.add(rx)
solver = PathSolver()

paths = solver(scene, max_depth=5)
scene.preview(paths=paths, show_devices=True)

Drag to rotate, scroll to zoom. The lines are the traced propagation paths.

## 2. What the RAN actually receives

The paths are turned into a tapped delay line. These taps are the channel the OCUDU DU
convolves onto every transmitted sample.

In [ ]:
def trace(position):
    """Traces one position and returns (taps, path_gain_dB, nof_paths)."""
    rx.position = [float(v) for v in position]
    p = solver(scene, max_depth=5)
    a, _ = p.cir(out_type="numpy")
    a = np.squeeze(np.asarray(a))
    nof_paths = int((np.abs(a) > 1e-12).sum()) if a.size else 0
    gain_db = float(10 * np.log10(np.sum(np.abs(a) ** 2) + 1e-30))
    t = p.taps(bandwidth=SRATE, l_min=0, l_max=NUM_TAPS - 1, sampling_frequency=SRATE,
               normalize=True, normalize_delays=True, out_type="numpy")
    return np.squeeze(np.asarray(t)).reshape(-1)[:NUM_TAPS], gain_db, nof_paths


taps_near, gain_near, n_near = trace(RX_START)
taps_far, gain_far, n_far = trace(RX_END)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), sharey=True)
for a_, t_, title, g, n in ((ax[0], taps_near, "UE near the gNB", gain_near, n_near),
                            (ax[1], taps_far, "UE far down the street", gain_far, n_far)):
    a_.stem(np.abs(t_))
    a_.set_title(f"{title}\npath gain {g:.1f} dB, {n} paths")
    a_.set_xlabel("tap")
    a_.grid(alpha=0.3)
ax[0].set_ylabel("|h|")
plt.tight_layout()
plt.show()

## 3. The route

The receiver is stepped along a street. Each position is re-traced, which is exactly what
`live_channel.py` streams to the running DU.

In [ ]:
STEPS = 30
start, end = np.array(RX_START), np.array(RX_END)
route = [start + (i / (STEPS - 1)) * (end - start) for i in range(STEPS)]

gains, counts, ys = [], [], []
for pos in route:
    _, g, n = trace(pos)
    gains.append(g)
    counts.append(n)
    ys.append(pos[1])

fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.plot(ys, gains, "o-", color="tab:blue", label="path gain")
ax1.set_xlabel("UE position along the street (m)")
ax1.set_ylabel("path gain (dB)", color="tab:blue")
ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.bar(ys, counts, width=5, alpha=0.25, color="tab:orange")
ax2.set_ylabel("number of paths", color="tab:orange")
plt.title("Ray-traced propagation along the route")
plt.tight_layout()
plt.show()

The dips are buildings shadowing the receiver, and the recoveries are the receiver clearing
them. The RAN reports the same shape as a falling and recovering SINR.

## 4. Rendered route

Frames rendered by `utils/sionna/render_route.py`, played as a video.

In [ ]:
from IPython.display import Video, display

video = os.path.join(REPO, "artifacts/demo_video/route.mp4")
if os.path.exists(video):
    display(Video(video, embed=True, width=900))
else:
    print(f"{video} not found. Generate it with:\n"
          f"  python utils/sionna/render_route.py --out-dir artifacts/demo_video \\\n"
          f"      --rx-start 20 40 1.5 --rx-end 20 300 1.5 --steps 30\n"
          f"  ffmpeg -y -framerate 5 -i artifacts/demo_video/frame_%03d.png \\\n"
          f"      -pix_fmt yuv420p artifacts/demo_video/route.mp4")

In [ ]:
from IPython.display import Image

frames = os.path.join(REPO, "artifacts/demo_video")
first, last = os.path.join(frames, "frame_000.png"), os.path.join(frames, "frame_029.png")
if os.path.exists(first) and os.path.exists(last):
    print("Near the gNB: a dense bundle of paths")
    display(Image(first, width=760))
    print("Far down the street: a single path bending around a building")
    display(Image(last, width=760))

## 5. Coverage map

Where the cell reaches, computed over the whole scene rather than a single route.

In [ ]:
from sionna.rt import RadioMapSolver

rm = RadioMapSolver()(scene, max_depth=5, cell_size=(8.0, 8.0), samples_per_tx=10**6)
scene.preview(radio_map=rm, show_devices=True)

## Feeding this to the RAN

The same trace is streamed to a running DU with

```bash
python -u utils/sionna/live_channel.py --bind tcp://127.0.0.1:5566 \
    --route --steps 12 --loop-route --interval 3.0 \
    --rx-start 20 40 1.5 --rx-end 20 260 1.5
```

The DU applies each update without restarting, and the UE reports the change as SINR.